# 🏆 Football Prediction App — 03: Model Training

**Amaç:** Feature matrisi üzerinde model eğitmek, değerlendirmek ve 2026 grup maçları için tahminler üretmek.

---
### Notebook bölümleri
1. Import & veri yükleme
2. Time-based train / valid / test split
3. Preprocessing pipeline
4. Baseline: Logistic Regression (H/D/A)
5. Poisson expected-goals modeli
6. Model değerlendirme (log loss, Brier score, kalibrasyon)
7. 2026 fikstür tahminleri
8. Skor dağılımı & market metrikleri
9. Model kaydetme

---
## 1. Import & Veri Yükleme

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import poisson

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    log_loss, brier_score_loss,
    classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from config import FILES, BASE_GOAL_RATE

PROCESSED_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
MODEL_DIR     = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
hist_path   = os.path.join(PROCESSED_DIR, 'features_historical.csv')
future_path = os.path.join(PROCESSED_DIR, 'features_2026_fixtures.csv')

if not os.path.isfile(hist_path):
    raise FileNotFoundError(
        "features_historical.csv bulunamadı.\n"
        "Önce 02_feature_engineering.ipynb'i çalıştırın."
    )

hist   = pd.read_csv(hist_path,   parse_dates=['date'])
future = pd.read_csv(future_path, parse_dates=['date_utc'])

print(f"Tarihi veri  : {hist.shape}")
print(f"2026 fikstür : {future.shape}")
print(f"\nHedef dağılımı:")
print(hist['result'].value_counts().to_string())

---
## 2. Time-Based Train / Valid / Test Split

Gelecekten geçmişi tahmin etmek anlamsız — split zamana göre yapılmalı.

| Küme | Tarih aralığı | Kullanım |
|---|---|---|
| **train** | 2000 – 2017 | Model eğitimi |
| **valid** | 2018 – 2021 | Hiperparametre ayarı |
| **test**  | 2022 – bugün | Son değerlendirme |

In [ ]:
train = hist[hist['date'] <  '2018-01-01'].copy()
valid = hist[(hist['date'] >= '2018-01-01') & (hist['date'] < '2022-01-01')].copy()
test  = hist[hist['date'] >= '2022-01-01'].copy()

print(f"Train : {len(train):>6,} maç  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Valid : {len(valid):>6,} maç  ({valid['date'].min().date()} → {valid['date'].max().date()})")
print(f"Test  : {len(test):>6,} maç  ({test['date'].min().date()} → {test['date'].max().date()})")

# Hedef encoding: H=0, D=1, A=2
LABEL_MAP = {'H': 0, 'D': 1, 'A': 2}
INV_MAP   = {v: k for k, v in LABEL_MAP.items()}

for df in [train, valid, test]:
    df['target'] = df['result'].map(LABEL_MAP)

---
## 3. Preprocessing Pipeline

- Eksik değerler → **median imputation**
- Sayısal feature'lar → **StandardScaler**

In [ ]:
# Model için kullanılacak feature sütunları
FEATURE_COLS = [
    'elo_diff',           # En güçlü sinyal
    'form_diff',          # Ağırlıklı form farkı
    'weighted_form_home',
    'weighted_form_away',
    'attack_home',
    'defense_home',
    'attack_away',
    'defense_away',
    'neutral',
    'tournament_weight',
]

def get_XY(df):
    X = df[FEATURE_COLS].copy()
    y = df['target'].values
    return X, y

X_train, y_train = get_XY(train)
X_valid, y_valid = get_XY(valid)
X_test,  y_test  = get_XY(test)

# Preprocessing pipeline
preproc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

X_train_t = preproc.fit_transform(X_train)
X_valid_t = preproc.transform(X_valid)
X_test_t  = preproc.transform(X_test)

print(f"Train  shape: {X_train_t.shape}")
print(f"Valid  shape: {X_valid_t.shape}")
print(f"Test   shape: {X_test_t.shape}")
print(f"NaN in train : {np.isnan(X_train_t).sum()}  (imputation sonrası)")

---
## 4. Baseline: Logistic Regression (H / D / A)

Hızlı, yorumlanabilir ve futbol tahmininde güçlü bir baseline.

In [ ]:
lr_model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=500,
    C=1.0,
    class_weight='balanced',
    random_state=42
)
lr_model.fit(X_train_t, y_train)

# Validation metrikleri
y_valid_prob = lr_model.predict_proba(X_valid_t)
y_valid_pred = lr_model.predict(X_valid_t)

val_logloss = log_loss(y_valid, y_valid_prob)
val_acc     = (y_valid_pred == y_valid).mean()

# Brier score — her sınıf için ayrı
brier_scores = {}
for i, label in enumerate(['H', 'D', 'A']):
    brier_scores[label] = brier_score_loss(
        (y_valid == i).astype(int),
        y_valid_prob[:, i]
    )

print("LOGISTIC REGRESSION — VALIDATION")
print("=" * 40)
print(f"  Log Loss        : {val_logloss:.4f}")
print(f"  Accuracy        : {val_acc*100:.1f}%")
print(f"  Brier (H)       : {brier_scores['H']:.4f}")
print(f"  Brier (D)       : {brier_scores['D']:.4f}")
print(f"  Brier (A)       : {brier_scores['A']:.4f}")

In [ ]:
# Feature önem katsayıları
coef_df = pd.DataFrame(
    lr_model.coef_,
    index=['H_coef', 'D_coef', 'A_coef'],
    columns=FEATURE_COLS
).T

print("Feature katsayıları (Logistic Regression):")
display(coef_df.round(3))

# Görsel
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(FEATURE_COLS))
width = 0.25
colors = ['steelblue', 'gray', 'coral']
for i, (col, color) in enumerate(zip(coef_df.columns, colors)):
    ax.bar(x + i*width, coef_df[col], width, label=col, color=color, alpha=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels(FEATURE_COLS, rotation=35, ha='right')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Feature Katsayıları')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Poisson Expected-Goals Modeli

Her takım için beklenen gol sayısı (`lambda`) hesaplanır.  
Poisson dağılımıyla skor matrisinden olasılıklar türetilir.

```
lambda_home = BASE_GOAL_RATE × attack_home × defense_away × context
lambda_away = BASE_GOAL_RATE × attack_away × defense_home × context
```

In [ ]:
# Poisson lambda modeli: Ridge regression ile gol sayısı tahmini
POISSON_FEATURES = [
    'elo_diff', 'attack_home', 'defense_away',
    'neutral', 'tournament_weight'
]

imputer_p = SimpleImputer(strategy='median')
scaler_p  = StandardScaler()

X_p_train = imputer_p.fit_transform(train[POISSON_FEATURES])
X_p_train = scaler_p.fit_transform(X_p_train)
X_p_valid = scaler_p.transform(imputer_p.transform(valid[POISSON_FEATURES]))

# Home gol modeli
home_goal_model = Ridge(alpha=1.0)
home_goal_model.fit(X_p_train, train['home_score'].clip(0, 8))

# Away gol modeli — elo farkı tersine çevrilir
away_features = POISSON_FEATURES.copy()
away_features[0] = 'elo_diff'  # away için -elo_diff kullanacağız

X_p_train_away = X_p_train.copy()
X_p_train_away[:, 0] = -X_p_train_away[:, 0]  # elo farkını ters çevir

away_features_alt = ['attack_away', 'defense_home', 'neutral', 'tournament_weight']
X_pa_train = imputer_p.fit_transform(train[away_features_alt + ['elo_diff']].assign(
    elo_diff=-train['elo_diff']
)[['elo_diff'] + away_features_alt])
X_pa_train = scaler_p.fit_transform(X_pa_train)
X_pa_valid = scaler_p.transform(imputer_p.transform(
    valid[away_features_alt + ['elo_diff']].assign(
        elo_diff=-valid['elo_diff']
    )[['elo_diff'] + away_features_alt]
))

away_goal_model = Ridge(alpha=1.0)
away_goal_model.fit(X_pa_train, train['away_score'].clip(0, 8))

# Validation MAE
pred_home = home_goal_model.predict(X_p_valid).clip(0.3, 5)
pred_away = away_goal_model.predict(X_pa_valid).clip(0.3, 5)

mae_home = np.mean(np.abs(pred_home - valid['home_score']))
mae_away = np.mean(np.abs(pred_away - valid['away_score']))
print(f"Poisson lambda — Validation MAE:")
print(f"  Home goals MAE: {mae_home:.3f}")
print(f"  Away goals MAE: {mae_away:.3f}")

In [ ]:
MAX_GOALS = 8  # Skor matrisinde göz önünde bulundurulan maksimum gol

def poisson_match_probs(lambda_home: float, lambda_away: float, max_goals: int = MAX_GOALS) -> dict:
    """
    Poisson dağılımından maç olasılıklarını hesaplar.
    Döner: p_home, p_draw, p_away, scoreline_probs, over_2_5, btts
    """
    lambda_home = max(0.1, lambda_home)
    lambda_away = max(0.1, lambda_away)

    # Skor matrisini oluştur
    goals = np.arange(0, max_goals + 1)
    p_home_goals = poisson.pmf(goals, lambda_home)
    p_away_goals = poisson.pmf(goals, lambda_away)
    score_matrix = np.outer(p_home_goals, p_away_goals)  # [home_goals, away_goals]

    p_home = np.tril(score_matrix, -1).sum()  # home > away
    p_draw = np.trace(score_matrix)           # home == away
    p_away = np.triu(score_matrix, 1).sum()   # away > home

    # Normalize
    total = p_home + p_draw + p_away
    p_home /= total; p_draw /= total; p_away /= total

    # Over 2.5
    over_2_5 = sum(
        score_matrix[h, a]
        for h in goals for a in goals
        if h + a > 2.5
    )

    # Both Teams to Score
    btts = sum(
        score_matrix[h, a]
        for h in goals for a in goals
        if h > 0 and a > 0
    )

    # En olası 5 skor
    scorelines = [
        (int(h), int(a), float(score_matrix[h, a]))
        for h in goals for a in goals
    ]
    top_scores = sorted(scorelines, key=lambda x: -x[2])[:5]

    return {
        'p_home'       : round(p_home, 4),
        'p_draw'       : round(p_draw, 4),
        'p_away'       : round(p_away, 4),
        'lambda_home'  : round(lambda_home, 3),
        'lambda_away'  : round(lambda_away, 3),
        'over_2_5'     : round(over_2_5, 4),
        'btts'         : round(btts, 4),
        'top_scorelines': top_scores,
    }

# Test
test_result = poisson_match_probs(1.78, 0.94)
print("Brazil vs Morocco (λ_home=1.78, λ_away=0.94):")
for k, v in test_result.items():
    if k != 'top_scorelines':
        print(f"  {k:<15}: {v}")
print("  En olası skorlar:")
for h, a, p in test_result['top_scorelines']:
    print(f"    {h}-{a}  →  {p*100:.1f}%")

---
## 6. Model Değerlendirme

In [ ]:
# Test seti üzerinde final değerlendirme
X_test_t_lr = preproc.transform(test[FEATURE_COLS])
y_test_prob  = lr_model.predict_proba(X_test_t_lr)
y_test_pred  = lr_model.predict(X_test_t_lr)

test_logloss = log_loss(y_test, y_test_prob)
test_acc     = (y_test_pred == y_test).mean()

test_brier = {}
for i, label in enumerate(['H', 'D', 'A']):
    test_brier[label] = brier_score_loss(
        (y_test == i).astype(int), y_test_prob[:, i]
    )

print("LOGISTIC REGRESSION — TEST SONUÇLARI")
print("=" * 42)
print(f"  Log Loss    : {test_logloss:.4f}")
print(f"  Accuracy    : {test_acc*100:.1f}%")
print(f"  Brier (H)   : {test_brier['H']:.4f}")
print(f"  Brier (D)   : {test_brier['D']:.4f}")
print(f"  Brier (A)   : {test_brier['A']:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=['H','D','A']))

In [ ]:
# Kalibrasyon grafiği + Confusion matrix
fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2)

# --- Kalibrasyon ---
ax1 = fig.add_subplot(gs[0])
colors = ['steelblue', 'gray', 'coral']
for i, (label, color) in enumerate(zip(['H', 'D', 'A'], colors)):
    y_bin = (y_test == i).astype(int)
    fraction_pos, mean_pred = calibration_curve(y_bin, y_test_prob[:, i], n_bins=10)
    ax1.plot(mean_pred, fraction_pos, marker='o', color=color,
             label=f'{label} sınıfı', linewidth=1.5)
ax1.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Mükemmel kalibrasyon')
ax1.set_xlabel('Tahmin edilen olasılık')
ax1.set_ylabel('Gerçekleşme oranı')
ax1.set_title('Kalibrasyon Eğrisi (Test)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- Confusion Matrix ---
ax2 = fig.add_subplot(gs[1])
cm = confusion_matrix(y_test, y_test_pred)
im = ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0,1,2]); ax2.set_yticks([0,1,2])
ax2.set_xticklabels(['H','D','A']); ax2.set_yticklabels(['H','D','A'])
ax2.set_xlabel('Tahmin'); ax2.set_ylabel('Gerçek')
ax2.set_title('Confusion Matrix (Test)')
for r in range(3):
    for c in range(3):
        ax2.text(c, r, str(cm[r, c]), ha='center', va='center',
                 color='white' if cm[r, c] > cm.max()/2 else 'black', fontsize=12)
plt.colorbar(im, ax=ax2)
plt.tight_layout()
plt.show()

---
## 7. 2026 Fikstür Tahminleri

In [ ]:
# Logistic model olasılıkları
future_avail = [c for c in FEATURE_COLS if c in future.columns]
X_future = future[FEATURE_COLS].copy()
X_future_t = preproc.transform(X_future)
future_probs = lr_model.predict_proba(X_future_t)

future['p_home_lr'] = future_probs[:, 0]
future['p_draw_lr'] = future_probs[:, 1]
future['p_away_lr'] = future_probs[:, 2]

print(f"Logistic tahminler eklendi: {len(future)} maç")

In [ ]:
# Poisson tahminleri — lambda hesapla
poisson_rows = []

for _, row in future.iterrows():
    # lambda_home
    att_h  = row.get('attack_home', 1.0) or 1.0
    def_a  = row.get('defense_away', 1.0) or 1.0
    elo_d  = row.get('elo_diff', 0) or 0
    neutral = row.get('neutral', 1)
    home_ctx = 1.0 if neutral else 1.1  # ev avantajı nötr sahada yok

    lambda_home = BASE_GOAL_RATE * att_h * def_a * home_ctx
    lambda_home = max(0.3, min(lambda_home, 5.0))

    # lambda_away
    att_a  = row.get('attack_away', 1.0) or 1.0
    def_h  = row.get('defense_home', 1.0) or 1.0
    lambda_away = BASE_GOAL_RATE * att_a * def_h
    lambda_away = max(0.3, min(lambda_away, 5.0))

    # Elo farkını hafifçe ağırlıklandır
    elo_factor = 1 + np.clip(elo_d / 1000, -0.3, 0.3)
    lambda_home *= elo_factor
    lambda_away /= elo_factor

    probs = poisson_match_probs(lambda_home, lambda_away)
    poisson_rows.append({
        'match_id'      : row['match_id'],
        'p_home_poi'    : probs['p_home'],
        'p_draw_poi'    : probs['p_draw'],
        'p_away_poi'    : probs['p_away'],
        'lambda_home'   : probs['lambda_home'],
        'lambda_away'   : probs['lambda_away'],
        'over_2_5'      : probs['over_2_5'],
        'btts'          : probs['btts'],
        'top_scorelines': str(probs['top_scorelines']),
    })

poisson_df = pd.DataFrame(poisson_rows)
future = future.merge(poisson_df, on='match_id', how='left')

print(f"Poisson tahminler eklendi: {len(future)} maç")

In [ ]:
# Ensemble: LR ve Poisson ortalaması
W_LR = 0.5   # Logistic Regression ağırlığı
W_POI = 0.5  # Poisson ağırlığı

future['p_home'] = W_LR * future['p_home_lr'] + W_POI * future['p_home_poi']
future['p_draw'] = W_LR * future['p_draw_lr'] + W_POI * future['p_draw_poi']
future['p_away'] = W_LR * future['p_away_lr'] + W_POI * future['p_away_poi']

# Normalize
total = future['p_home'] + future['p_draw'] + future['p_away']
future['p_home'] = (future['p_home'] / total).round(4)
future['p_draw'] = (future['p_draw'] / total).round(4)
future['p_away'] = (future['p_away'] / total).round(4)

# Favori
def get_favourite(row):
    mx = max(row['p_home'], row['p_draw'], row['p_away'])
    if mx == row['p_home']: return row['home_team']
    if mx == row['p_draw']:  return 'Draw'
    return row['away_team']

future['favourite'] = future.apply(get_favourite, axis=1)

print("2026 GRUP MAÇLARI — TAHMİNLER")
print("=" * 80)
display_cols = ['group','home_team','away_team','p_home','p_draw','p_away',
                'lambda_home','lambda_away','over_2_5','btts','favourite','upset_risk']
available = [c for c in display_cols if c in future.columns]
display(future[available].sort_values('group').reset_index(drop=True))

---
## 8. Skor Dağılımı — Görsel Maç Kartı

In [ ]:
def plot_match_card(row, ax_bar=None, ax_score=None):
    """Bir maç için tahmin kartı çizer."""
    if ax_bar is None:
        fig, (ax_bar, ax_score) = plt.subplots(1, 2, figsize=(12, 4))

    home = row['home_team']
    away = row['away_team']
    title = f"{home} vs {away}"

    # Olasılık bar
    labels = [f"{home}\n({row['p_home']*100:.0f}%)",
              f"Beraberlik\n({row['p_draw']*100:.0f}%)",
              f"{away}\n({row['p_away']*100:.0f}%)"]
    values = [row['p_home'], row['p_draw'], row['p_away']]
    colors = ['steelblue', 'lightgray', 'coral']

    bars = ax_bar.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
    ax_bar.set_ylim(0, 1)
    ax_bar.set_title(title, fontsize=11, fontweight='bold')
    ax_bar.set_ylabel('Olasılık')
    ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%'))
    for bar, val in zip(bars, values):
        ax_bar.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                    f"{val*100:.1f}%", ha='center', va='bottom', fontsize=9)

    # Skor dağılımı (Poisson)
    lh = row.get('lambda_home', 1.2)
    la = row.get('lambda_away', 1.0)
    if pd.isna(lh): lh = 1.2
    if pd.isna(la): la = 1.0
    goals = np.arange(0, 6)
    ph = poisson.pmf(goals, lh)
    pa = poisson.pmf(goals, la)
    sm = np.outer(ph, pa)

    all_scores = [(int(h), int(a), sm[h, a]) for h in goals for a in goals]
    top5 = sorted(all_scores, key=lambda x: -x[2])[:5]

    sc_labels = [f"{h}-{a}" for h, a, _ in top5]
    sc_vals   = [p for _, _, p in top5]
    ax_score.bar(sc_labels, sc_vals, color='mediumpurple', alpha=0.8)
    ax_score.set_title(f'En Olası Skorlar  (xG: {lh:.2f} - {la:.2f})', fontsize=10)
    ax_score.set_ylabel('Olasılık')
    ax_score.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v*100:.1f}%'))

# İlk 6 maçı göster
sample_matches = future.head(6)
fig, axes = plt.subplots(6, 2, figsize=(13, 22))
for i, (_, row) in enumerate(sample_matches.iterrows()):
    plot_match_card(row, ax_bar=axes[i][0], ax_score=axes[i][1])
plt.tight_layout()
plt.show()

---
## 9. Model & Tahminleri Kaydet

In [ ]:
# Modelleri kaydet
joblib.dump(lr_model,         os.path.join(MODEL_DIR, 'lr_model.pkl'))
joblib.dump(preproc,          os.path.join(MODEL_DIR, 'preprocessor.pkl'))
joblib.dump(home_goal_model,  os.path.join(MODEL_DIR, 'home_goal_model.pkl'))
joblib.dump(away_goal_model,  os.path.join(MODEL_DIR, 'away_goal_model.pkl'))
joblib.dump(imputer_p,        os.path.join(MODEL_DIR, 'poisson_imputer.pkl'))
joblib.dump(scaler_p,         os.path.join(MODEL_DIR, 'poisson_scaler.pkl'))

print("Modeller kaydedildi:")
for f in os.listdir(MODEL_DIR):
    size = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024
    print(f"  {f:<35} {size:.1f} KB")

In [ ]:
# Tahminleri kaydet
pred_cols = [
    'match_id', 'group', 'date_utc', 'venue',
    'home_team', 'away_team',
    'elo_home', 'elo_away', 'elo_diff',
    'p_home', 'p_draw', 'p_away',
    'lambda_home', 'lambda_away',
    'over_2_5', 'btts',
    'top_scorelines', 'favourite', 'upset_risk', 'upset_label',
]
available_pred_cols = [c for c in pred_cols if c in future.columns]
predictions = future[available_pred_cols].copy()

pred_path = os.path.join(PROCESSED_DIR, 'predictions_latest.csv')
predictions.to_csv(pred_path, index=False)

print(f"✅  Tahminler kaydedildi → {pred_path}")
print(f"    {len(predictions)} maç, {len(predictions.columns)} kolon")
print()
display(predictions[['group','home_team','away_team','p_home','p_draw','p_away',
                       'lambda_home','lambda_away','over_2_5','favourite']].head(10))

---
## 10. Sonraki Adım

> **`04_world_cup_simulation.ipynb`** →  
> - `predictions_latest.csv` yükle  
> - Monte Carlo simülasyonu (10.000x)  
> - Grup tablosu hesaplama  
> - Knockout slotlarını doldur  
> - Her takım için şampiyonluk / aşama olasılıkları  
> - `simulation_latest.csv` çıktısı